In [ ]:
import numpy as np
import ot_utils
import comparison_utils
from tqdm import tqdm
import math
import matplotlib.pyplot as plt

n_classes = 8
labels = np.repeat(np.arange(8), 1000)

w2_block_files = []
ps_block_files = []

for i in range(n_classes):
    for j in range(i, n_classes):
        w2_block_files.append(f"../mini_w2/block_{i}_{j}.npy")

w2_matrix = ot_utils.combine_distance_blocks(w2_block_files)
ps_matrix = np.load("../mini_ps/8k_ps_dist_matrix.npy")

In [2]:
w2_rel_rep, ps_rel_rep = comparison_utils.relative_rep(
    w2_matrix, ps_matrix, labels, n_classes, anchors_per_class=10, seed=1
)

In [8]:
print(
    [
        np.min(np.linalg.norm(ps_rel_rep[i * 1000 : (i + 1) * 1000], axis=1))
        for i in range(8)
    ]
)

print(
    [
        np.max(np.linalg.norm(ps_rel_rep[i * 1000 : (i + 1) * 1000], axis=1))
        for i in range(8)
    ]
)

[12.887759451036574, 12.884407199353067, 12.887257494488068, 12.892160124632637, 12.885838997368298, 12.888476154244772, 12.884439234651625, 12.890019325614752]
[71.26789408761566, 31.313059351355776, 28.436468543103718, 29.261369325443688, 30.320329368010736, 31.624472316032893, 32.97619910236408, 30.551210599018408]


In [ ]:
print(
    [
        np.min(np.linalg.norm(w2_rel_rep[i * 1000 : (i + 1) * 1000], axis=1))
        for i in range(8)
    ]
)

print(
    [
        np.max(np.linalg.norm(w2_rel_rep[i * 1000 : (i + 1) * 1000], axis=1))
        for i in range(8)
    ]
)

[1.3257161982838186, 1.344412824977609, 1.4095745264047002, 1.3251477248843373, 1.3637910046594492, 1.349878548144445, 1.3511705628965704, 1.4655002073152663]
[1.7557631765549213, 3.2414365617787184, 3.111854389056055, 3.198006437678777, 3.355683662851248, 3.0334789036882017, 2.994724707874017, 3.3178245565123317]


In [ ]:
n_samples = 8000
w2_pairwise_euc_sim = np.zeros((n_samples, n_samples))
ps_pairwise_euc_sim = np.zeros((n_samples, n_samples))

i_pairs, j_pairs = np.triu_indices(n_samples, k=1)

for i, j in tqdm(zip(i_pairs, j_pairs), total=len(i_pairs)):
    w2_pairwise_euc_sim[i, j] = comparison_utils.eud_sim_2(w2_rel_rep[i], w2_rel_rep[j])
    w2_pairwise_euc_sim[j, i] = w2_pairwise_euc_sim[i, j]

    ps_pairwise_euc_sim[i, j] = comparison_utils.eud_sim_2(ps_rel_rep[i], ps_rel_rep[j])
    ps_pairwise_euc_sim[j, i] = ps_pairwise_euc_sim[i, j]

100%|██████████| 31996000/31996000 [02:07<00:00, 251772.78it/s]


In [5]:
qcd_i, qcd_j = np.triu_indices(1000, k=1)

w2_qcd_eud_sims = w2_pairwise_euc_sim[qcd_i, qcd_j]
ps_qcd_eud_sims = ps_pairwise_euc_sim[qcd_i, qcd_j]

print(w2_qcd_eud_sims.mean())
print(ps_qcd_eud_sims.mean())

0.835660722841994
0.12019002979147482


In [7]:
class_1_i = qcd_i + 1000
class_1_j = qcd_j + 1000

w2_class_1_eud_sims = w2_pairwise_euc_sim[class_1_i, class_1_j]
ps_class_1_eud_sims = ps_pairwise_euc_sim[class_1_i, class_1_j]

print(w2_class_1_eud_sims.mean())
print(ps_class_1_eud_sims.mean())

0.6918285906064402
0.2376123312962762


In [3]:
row_norms_w2 = np.linalg.norm(w2_rel_rep, axis=1, keepdims=True)
row_norms_ps = np.linalg.norm(ps_rel_rep, axis=1, keepdims=True)

normalized_w2_rel_rep = w2_rel_rep / row_norms_w2
normalized_ps_rel_rep = ps_rel_rep / row_norms_ps

In [ ]:
n_samples = 8000
w2_pairwise_euc_sim = np.zeros((n_samples, n_samples))
ps_pairwise_euc_sim = np.zeros((n_samples, n_samples))

w2_pairwise_cos_sim = np.zeros((n_samples, n_samples))
ps_pairwise_cos_sim = np.zeros((n_samples, n_samples))

i_pairs, j_pairs = np.triu_indices(n_samples, k=1)

for i, j in tqdm(zip(i_pairs, j_pairs), total=len(i_pairs)):
    w2_pairwise_euc_sim[i, j] = comparison_utils.eud_sim(
        normalized_w2_rel_rep[i], normalized_w2_rel_rep[j]
    )
    w2_pairwise_euc_sim[j, i] = w2_pairwise_euc_sim[i, j]

    ps_pairwise_euc_sim[i, j] = comparison_utils.eud_sim(
        normalized_ps_rel_rep[i], normalized_ps_rel_rep[j]
    )
    ps_pairwise_euc_sim[j, i] = ps_pairwise_euc_sim[i, j]

    w2_pairwise_cos_sim[i, j] = comparison_utils.cos_sim(
        normalized_w2_rel_rep[i], normalized_ps_rel_rep[j]
    )
    w2_pairwise_cos_sim[j, i] = w2_pairwise_cos_sim[i, j]

    ps_pairwise_cos_sim[i, j] = comparison_utils.cos_sim(
        normalized_w2_rel_rep[i], normalized_ps_rel_rep[j]
    )
    ps_pairwise_cos_sim[j, i] = ps_pairwise_cos_sim[i, j]

100%|██████████| 31996000/31996000 [03:16<00:00, 162940.98it/s]


In [5]:
qcd_i, qcd_j = np.triu_indices(1000, k=1)

w2_qcd_eud_sims = w2_pairwise_euc_sim[qcd_i, qcd_j]
ps_qcd_eud_sims = ps_pairwise_euc_sim[qcd_i, qcd_j]

w2_qcd_cos_sims = w2_pairwise_cos_sim[qcd_i, qcd_j]
ps_qcd_cos_sims = ps_pairwise_cos_sim[qcd_i, qcd_j]

print(w2_qcd_eud_sims.mean())
print(ps_qcd_eud_sims.mean())
print(w2_qcd_cos_sims.mean())
print(ps_qcd_cos_sims.mean())

0.9223045448908347
0.7992246791708105
0.9524018391216901
0.9524018391216901


In [14]:
class_1_i = qcd_i + 7000
class_1_j = qcd_j + 7000

w2_class_1_eud_sims = w2_pairwise_euc_sim[class_1_i, class_1_j]
ps_class_1_eud_sims = ps_pairwise_euc_sim[class_1_i, class_1_j]

print(w2_class_1_eud_sims.mean())
print(ps_class_1_eud_sims.mean())

0.8790395081489532
0.8731012909919307


In [ ]:
from sklearn.metrics import pairwise_distances

pairwise_eud_sims_w2_ps = pairwise_distances(
    normalized_w2_rel_rep,
    normalized_ps_rel_rep,
    metric=lambda u, v: comparison_utils.eud_sim(u, v),
    n_jobs=-1,  # use all CPUs to parallelize
)

pairwise_cos_sims_w2_ps = pairwise_distances(
    normalized_w2_rel_rep,
    normalized_ps_rel_rep,
    metric=lambda u, v: comparison_utils.cos_sim(u, v),
    n_jobs=-1,
)

print(pairwise_eud_sims_w2_ps.shape)

(8000, 8000)


In [7]:
rows, cols = np.indices((1000, 1000))

i = 0
rows = rows + 1000 * i
cols = cols + 1000 * i

w2_ps_qcd_eud_sims = pairwise_eud_sims_w2_ps[rows, cols]
w2_ps_qcd_cos_sims = pairwise_cos_sims_w2_ps[rows, cols]

print(w2_ps_qcd_eud_sims.mean())
print(w2_ps_qcd_cos_sims.mean())

0.8258293420550329
0.9517517193344941
